In [1]:
# ---
# Imports and Setup
# ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display options
pd.set_option("display.max_columns", 100)
sns.set_theme("notebook")

In [3]:
import os
os.getcwd()

'/Users/stefannhs/Projects/open-rec-sphere/notebooks'

In [5]:
# ---
# Imports and Setup
# ---

path = "../data/raw/expedia/"
df_train = pd.read_csv(path + "train.csv")
df_destinations = pd.read_csv(path + "destinations.csv")

In [23]:
# ---
# Overview
# ---

df_train.shape

(37670293, 24)

In [24]:
df_train.head()

,date_time,site_name,posa_continent,user_location_country,user_location_region,user_location_city,orig_destination_distance,user_id,is_mobile,is_package,channel,srch_ci,srch_co,srch_adults_cnt,srch_children_cnt,srch_rm_cnt,srch_destination_id,srch_destination_type_id,is_booking,cnt,hotel_continent,hotel_country,hotel_market,hotel_cluster
0,2014-08-11 07:46:59,2,3,66,348,48862,2234.2641,12,0,1,9,2014-08-27,2014-08-31,2,0,1,8250,1,0,3,2,50,628,1
1,2014-08-11 08:22:12,2,3,66,348,48862,2234.2641,12,0,1,9,2014-08-29,2014-09-02,2,0,1,8250,1,1,1,2,50,628,1
2,2014-08-11 08:24:33,2,3,66,348,48862,2234.2641,12,0,0,9,2014-08-29,2014-09-02,2,0,1,8250,1,0,1,2,50,628,1
3,2014-08-09 18:05:16,2,3,66,442,35390,913.1932,93,0,0,3,2014-11-23,2014-11-28,2,0,1,14984,1,0,1,2,50,1457,80
4,2014-08-09 18:08:18,2,3,66,442,35390,913.6259,93,0,0,3,2014-11-23,2014-11-28,2,0,1,14984,1,0,1,2,50,1457,21


In [25]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37670293 entries, 0 to 37670292
Data columns (total 24 columns):
 #   Column                     Dtype  
---  ------                     -----  
 0   date_time                  object 
 1   site_name                  int64  
 2   posa_continent             int64  
 3   user_location_country      int64  
 4   user_location_region       int64  
 5   user_location_city         int64  
 6   orig_destination_distance  float64
 7   user_id                    int64  
 8   is_mobile                  int64  
 9   is_package                 int64  
 10  channel                    int64  
 11  srch_ci                    object 
 12  srch_co                    object 
 13  srch_adults_cnt            int64  
 14  srch_children_cnt          int64  
 15  srch_rm_cnt                int64  
 16  srch_destination_id        int64  
 17  srch_destination_type_id   int64  
 18  is_booking                 int64  
 19  cnt                        int64  
 20  

## Column Classification Overview

Before diving into deeper analysis, we define and group the columns in `train.csv` based on their data types and functional roles.

This allows us to:
- Structure our EDA by column type
- Apply targeted preprocessing strategies later
- Document our assumptions about how the data behaves

We'll classify columns as:
- **Numerical**
- **Categorical**
- **Binary / Boolean**
- **Datetime**
- **ID / Structural**

In [32]:
# --- Define column groups based on semantics and dtype ---

# Continuous numerical features
numerical_cols = [
    "orig_destination_distance",
    "srch_adults_cnt",
    "srch_children_cnt",
    "srch_rm_cnt",
    "cnt",
]

# Categorical features (IDs that represent groups or categories)
categorical_cols = [
    "site_name",
    "posa_continent",
    "user_location_country",
    "user_location_region",
    "user_location_city",
    "channel",
    "hotel_continent",
    "hotel_country",
    "hotel_market",
]

# Binary flags (0 or 1)
binary_cols = [
    "is_mobile",
    "is_package",
    "is_booking",
]

# Date and time features
datetime_cols = [
    "date_time",
    "srch_ci",
    "srch_co",
]

# IDs and label (used for joining, indexing, and modeling)
id_cols = [
    "user_id",
    "srch_destination_id",
    "srch_destination_type_id",
    "hotel_cluster", # TARGET LABEL
]

In [34]:
df_train[numerical_cols].describe()

,orig_destination_distance,srch_adults_cnt,srch_children_cnt,srch_rm_cnt,cnt
count,2.414529e+07,3.767029e+07,3.767029e+07,3.767029e+07,3.767029e+07
mean,1.970090e+03,2.024296e+00,3.321222e-01,1.112663e+00,1.483384e+00
std,2.232442e+03,9.116678e-01,7.314981e-01,4.591155e-01,1.219776e+00
min,5.600000e-03,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00
25%,3.131670e+02,2.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00
50%,1.140491e+03,2.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00
75%,2.552599e+03,2.000000e+00,0.000000e+00,1.000000e+00,2.000000e+00
max,1.240790e+04,9.000000e+00,9.000000e+00,8.000000e+00,2.690000e+02


In [39]:
for col in categorical_cols:
    n_unique = df_train[col].nunique()
    print(f"{col}: {n_unique} unique values")


site_name: 45 unique values
posa_continent: 5 unique values
user_location_country: 237 unique values
user_location_region: 1008 unique values
user_location_city: 50447 unique values
channel: 11 unique values
hotel_continent: 7 unique values
hotel_country: 213 unique values
hotel_market: 2118 unique values


In [40]:
for col in categorical_cols:
    print(f"\n📊 {col} — Top 5 categories:")
    print(df_train[col].value_counts().head(5))


📊 site_name — Top 5 categories:
site_name
2     23790351
11     2605866
24     2363595
37     2013818
34     1784564
Name: count, dtype: int64

📊 posa_continent — Top 5 categories:
posa_continent
3    28240462
1     4458076
2     3515919
4     1190726
0      265110
Name: count, dtype: int64

📊 user_location_country — Top 5 categories:
user_location_country
66     20346844
205     4188283
3       2212572
69      1931466
77       948722
Name: count, dtype: int64

📊 user_location_region — Top 5 categories:
user_location_region
174    4132663
348    1873377
354    1708208
442    1432740
220    1365143
Name: count, dtype: int64

📊 user_location_city — Top 5 categories:
user_location_city
5703     762261
48862    617937
25315    400451
24103    394731
36086    354610
Name: count, dtype: int64

📊 channel — Top 5 categories:
channel
9    20881299
0     4685201
1     3819309
2     2966352
5     2326077
Name: count, dtype: int64

📊 hotel_continent — Top 5 categories:
hotel_continent
2    197776

## 1. Dataset Familiarization

### Q: What does each row in train.csv represent — a search, a session, or a single interaction?

From the dataset description:

We're looking at logs of customer behavior, which include:
- what customers searched for,
- how they interacted with search results (click/book),
- whether or not the search result was a travel package.

Training data includes all the users in the logs, including both click events and booking events.

Here is the flow that generates the training data:

1. User searches for hotels (with search parameters)
2. Expedia returns a set of hotel options
3. Each hotel option is a row in the training dataset
4. A user can interact with an option (e.g. click, book) - this is tracked

### Q: What is hotel_cluster? Why are there 100 of them?

#### What is hotel_cluster?

`hotel_cluster` is a **categorical label** assigned to each hotel in the dataset, created by **Expedia’s internal clustering algorithms**. These clusters group similar hotels based on:

- Historical booking behavior
- Price levels
- Star ratings
- Geographic proximity (e.g., distance to city center)
- Possibly other latent features

This clustering allows Expedia to **reduce the complexity** of modeling individual hotels and instead focus on groups that behave similarly in terms of user preferences and booking patterns.

#### Why are there 100 of them?

We assume Expedia originally had **many more clusters**, but for the purposes of the Kaggle competition, they:

- **Randomly sampled 100 hotel clusters** from their full set
- Designed the challenge to be a **100-class classification problem**

This constraint makes the problem more tractable for participants, while still simulating a real-world multi-class recommendation scenario.



### Q: Do all columns have intuitive meanings? What about `user_location_city` vs `orig_destination_distance`?

#### General observation

While the column names are fairly descriptive, they are **internally-defined abstractions** rather than human-readable values — so **some semantics are hidden** behind ID mappings (e.g., `user_location_city`, `hotel_market`, `srch_destination_id`). You can interpret many of them through inference and exploratory analysis, but you don’t get explicit labels like "Amsterdam" or "New York".

Expedia did provide a data field description, but **interpretation still requires domain reasoning** — especially for interactions between location, time, and booking behavior.

#### Focus: `user_location_city` vs `orig_destination_distance`

|  Feature |  Description  | Data Type  |
|----------|---------------|------------|
|  `user_location_city`  |  The ID of the city the customer is located  |  int  |
|  `orig_destination_distance`  |  Physical distance between a hotel and a customer at the time of search. A null means the distance could not be calculated  |  double  |

##### Why it matters?
`user_location_city` is a **categorical variable**, and useful for understanding **home-base** or regional trends (e.g. Dutch users booking hotels in Spain during summer).

`orig_destination_distance` is a **continuous numerical variable**, and useful for identifying domestic vs international trips, travel intent, or travel planning behavior.

Their relationship can reveal latent clusters in travel behavior — like whether short-distance searches tend to be booked more often (weekend trips vs intercontinental vacations).




### Q: How many columns in destinations.csv? What's the shape and dtype?

In [22]:
df_destinations.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62106 entries, 0 to 62105
Columns: 150 entries, srch_destination_id to d149
dtypes: float64(149), int64(1)
memory usage: 71.1 MB


The destinations.csv file contains:

- **62,106 rows** — each representing a unique `srch_destination_id`
- **150 columns** in total:
    - `srch_destination_id` (type: `int64`)
    - 149 feature columns named d1 to d149 (type: `float64`)
- The dataset uses **~71MB of memory** in its raw form

These 149 $d_{n}$ columns are dense numerical embeddings or latent features that likely capture characteristics of the destination.

Since they’re floats, they could represent:
- Embedding vectors from historical user behavior
- Aggregated signals (e.g., popularity, price profile, geographic features)

## 2. Data Integrity Checks
🧪 Hints:

Are there any missing values? If so, are they ignorable or structural?

Check for duplicates — are there repeated srch_ids or users?

Are all hotel clusters between 0–99? Any unexpected values?

What’s the distribution of click_bool, booking_bool? Class imbalance?

